# Incident Response and Recovery

> **The story.** In November 1988, the Morris worm disrupted thousands of internet-connected systems and exposed how poorly organizations coordinated technical response across institutional boundaries. DARPA funded the CERT Coordination Center soon after, helping formalize incident coordination as a discipline rather than an improvised debugging session. Riverside's AI assistant adds newer failure surfaces, but the operational lesson is unchanged: stabilize shared risk before chasing an elegant diagnosis.
>
> **Where you are.** The FDE route has already defined a frozen Riverside engagement, strict identity and policy boundaries, staged rollout gates, and named owners. Six canonical incidents now stress those decisions, and one exercise-only retrieval variation tests whether you can separate index failure from model behavior.
>
> **Notation.** $I$ - incident; $S$ - severity; $E$ - preserved evidence set; $H_i$ - competing causal hypothesis; $G_j$ - regression gate; $R$ - bounded re-enablement decision; all times are UTC.

> **Validation status:** This notebook executed successfully against the synthetic incident fixtures and produced the documented containment and re-enablement outcomes. Outputs were then cleared, so the committed notebook contains no retained incident measurements. No live incident, provider operation, customer communication, legal decision, or production recovery was performed.

> **What you finished last time:** staged rollout established abort conditions, bounded cohorts, rollback targets, and decision owners.
> **What this notebook delivers:** a completed synthetic incident record, redacted communication, regression package, re-enablement decision, and postmortem.
> **Prerequisite for the next notebook:** handoff receives verified runbooks, alert ownership, temporary-control expiry, and corrective actions.

## 0 - The Challenge

> **The mission**: Riverside House - contain and recover seven synthetic failure domains without unauthorized disclosure, duplicate workflow commits, residency violations, or unsupported customer claims.

**What we know so far:**

- The frozen case names six canonical incidents, eight risks, six rollout cohorts, and global abort conditions.
- A forbidden access, autonomous protected action, duplicate commit, or `SEV-1` incident stops rollout.
- **But a rollback target does not tell you how to preserve evidence, classify uncertain impact, communicate safely, or approve recovery.**

**What's blocking us:**

`INC-RIV-003` proves a disabled contractor can retrieve a restricted EU manuscript chunk. The unsafe route is still conceptually live, the blast radius is unknown, and raw logs may contain restricted fields. Debugging first would allow more exposure; copying raw evidence into chat would create another incident.

**What this chapter unlocks:**

A reviewable path from declaration through bounded re-enablement, with evidence-backed severity, redacted updates, causal tests, regression gates, approvals, and postmortem actions.

| Sub-topic | Coverage | Why |
|---|---|---|
| Containment, evidence, severity, communication | Built | Core FDE response decisions |
| Seven-domain causal triage | Built | Prevents component blame by intuition |
| Regression and re-enablement gates | Built | Recovery is a new exposure decision |
| Postmortem action verification | Built | A document alone does not prevent recurrence |
| Forensic imaging and malware analysis | Named only | Requires specialist tooling, authority, and handling procedures |
| Jurisdictional notification law | Named only | Must be decided by authorized legal/privacy owners |
| Live cloud/provider operations | Named only | This chapter is local and synthetic; provider behavior remains unvalidated |

```mermaid
flowchart LR
    A["Symptom detected"] --> B{"Unsafe exposure active?"}
    B -->|Yes or unknown| C["Contain narrow path"]
    B -->|No, proven| D["Preserve state"]
    C --> D
    D --> E["Classify plausible impact"]
    E --> F["Communicate facts and unknowns"]
    F --> G["Test competing causes"]
    G --> H["Remediate and regress"]
    H --> I{"Evidence plus authority?"}
    I -->|No| C
    I -->|Yes| J["Bounded re-enablement"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style I fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style J fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

In [ ]:
# -- Load the static scenario contract ---------------------------------------
from pathlib import Path
import json
import re

import matplotlib.pyplot as plt
import pandas as pd

CHAPTER_DIR = Path.cwd()
SCENARIO_PATH = CHAPTER_DIR / "incidents" / "incident-scenarios-v1.json"
if not SCENARIO_PATH.exists():
    raise FileNotFoundError(
        "Run this notebook from learning/fde/07-incident-response-and-recovery."
    )

with SCENARIO_PATH.open(encoding="utf-8") as scenario_file:
    scenario_contract = json.load(scenario_file)

scenarios = scenario_contract["scenarios"]
scenario_by_id = {scenario["exercise_id"]: scenario for scenario in scenarios}

print(f"Loaded {len(scenarios)} synthetic scenarios from {SCENARIO_PATH.name}.")
print("STATIC NOTE: loading later will inspect fixtures; it will not create production evidence.")

**Predict:** Which contract defect is most dangerous?

1. The scenarios appear in a different order.
2. A scenario has no regression gates.
3. The exercise-only retrieval scenario is labeled canonical.

Choose before running the next cell. Missing gates are serious, but one defect turns invented teaching data into a false customer claim.

In [ ]:
# -- Verify scenario provenance and response fields -------------------------
required_domains = {
    "policy", "data", "retrieval", "model", "tool", "identity", "infrastructure"
}
required_fields = {
    "exercise_id", "source_kind", "domain", "initial_severity",
    "first_containment", "evidence_to_preserve", "causal_questions",
    "regression_gates", "reenablement_roles",
}
domains = {scenario["domain"] for scenario in scenarios}
missing_fields = {
    scenario["exercise_id"]: sorted(required_fields - set(scenario))
    for scenario in scenarios
    if required_fields - set(scenario)
}
retrieval = scenario_by_id["IRX-RIV-007"]
checks = {
    "all seven domains present": domains == required_domains,
    "all response fields present": not missing_fields,
    "retrieval provenance is honest": retrieval["source_kind"] == "exercise_only_derived",
    "retrieval has no canonical incident ID": retrieval["canonical_incident_id"] is None,
}
for label, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {label}")

actual_answer = "option 3" if checks["retrieval provenance is honest"] else "provenance failure"
print(f"Prediction resolved: {actual_answer} - invented customer evidence is the categorical error.")

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Copy fixture text into a status update and call it measured | Synthetic case text becomes an unsupported production claim |
| Right | Preserve `canonical_extension` versus `exercise_only_derived` | Reviewers can distinguish case facts from teaching variations |
| Wrong | Treat scenario order as identity | Reordering silently changes joins |
| Right | Join by stable exercise and canonical IDs | Provenance survives refactoring |

**Quick Health Check**

- Seven domains are present exactly once.
- Six exercises reference canonical incidents.
- Retrieval is explicitly exercise-only.
- Every exercise has containment, evidence, gates, and re-enablement roles.

The previous code cell runs these checks. The contract is ready; now the unsafe route has to stop.

## 1 · Declare and Contain First

Containment reduces exposure. Diagnosis explains exposure. Reversing those verbs keeps the incident active while the team debates architecture.

```mermaid
flowchart TD
    A["Detection or credible report"] --> B["Assign commander and scribe"]
    B --> C{"False allow, unsafe output, mutation, or residency risk?"}
    C -->|Yes or unknown| D["Stop affected route or action"]
    C -->|No, availability only| E["Use approved degraded mode"]
    D --> F["Freeze related changes"]
    E --> F
    F --> G["Record last known-good state"]
    G --> H["State next update and owner"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

The safest containment is usually narrower than teardown: disable one tenant route, pause writes, pin an index, revoke one identity, or require abstention for one question class. Keep authentication, authorization, redaction, residency, and audit fail-closed.

**Predict:** The former contractor false allow is detected. Which first action is defensible?

1. Keep serving while querying more traces.
2. Disable the EU route, revoke the identity, preserve evidence, and engage Security and Legal.
3. Delete the suspect index and rotate every identity.

In [ ]:
# -- Resolve containment and practice a bounded tool response ----------------
identity_incident = scenario_by_id["IRX-RIV-003"]
expected_containment = (
    "Disable the EU tenant route, revoke the identity, preserve access evidence, "
    "and engage Security and Legal."
)
prediction_confirmed = identity_incident["first_containment"] == expected_containment
print(f"{'PASS' if prediction_confirmed else 'FAIL'}: option 2 matches the frozen containment boundary.")

# CHANGE THIS: choose 'pause_writes' instead of 'disable_everything'.
containment_mode = "pause_writes"
tool_incident = scenario_by_id["IRX-RIV-005"]
bounded = containment_mode == "pause_writes" and "read-only assistance" in tool_incident["first_containment"]
print(f"{'PASS' if bounded else 'FAIL'}: mutating workflow exposure is stopped.")
print("Read-only assistance remains available only within existing authorization controls.")

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Debug before stopping a false allow | Every new request can expand impact |
| Right | Disable the affected route, then investigate | Exposure is bounded while evidence remains available |
| Wrong | Restore availability by bypassing policy or residency | Recovery creates a larger control failure |
| Right | Use only pre-approved degraded modes | Availability remains subordinate to authority |

**Quick Health Check**

1. Incident commander and scribe are assigned.
2. The unsafe path and preserved safe path are named.
3. Related deploy, index, policy, and data changes are frozen.
4. Last-known-good state and next update time are recorded.
5. Suspect state is preserved rather than deleted.

## 2 · Preserve Evidence and Build the Timeline

Evidence is useful only when you can explain what it is, where it came from, who handled it, which time range it covers, and what changed after collection. A pasted log fragment without provenance is a clue, not a defensible incident record.

```mermaid
flowchart LR
    A["Source state"] --> B["Approved collection"]
    B --> C["Content-free reference"]
    C --> D["Integrity and custody record"]
    D --> E["Restricted evidence store"]
    E --> F["Timeline fact or hypothesis"]
    F --> G{"Correction needed?"}
    G -->|Yes| H["Append superseding entry"]
    G -->|No| I["Retain immutable history"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style I fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

Record four clocks separately: when the system event occurred, when a signal observed it, when a human made a decision, and when an audience received an update. Collapsing them into one timestamp hides detection and communication delay.

In [ ]:
# -- Build an exercise-only UTC timeline visual ----------------------------
timeline_rows = [
    {"event": "Access decision", "event_time": "2026-07-16T07:22:40Z", "recorded_time": "2026-07-16T07:25:00Z", "kind": "FACT"},
    {"event": "Negative test alert", "event_time": "2026-07-16T07:25:00Z", "recorded_time": "2026-07-16T07:25:15Z", "kind": "FACT"},
    {"event": "EU route disabled", "event_time": "2026-07-16T07:29:00Z", "recorded_time": "2026-07-16T07:29:20Z", "kind": "DECISION"},
    {"event": "Evidence manifest opened", "event_time": "2026-07-16T07:31:00Z", "recorded_time": "2026-07-16T07:31:00Z", "kind": "EVIDENCE"},
    {"event": "First customer update due", "event_time": "2026-07-16T08:00:00Z", "recorded_time": "2026-07-16T08:00:00Z", "kind": "COMMUNICATION"},
]
timeline = pd.DataFrame(timeline_rows)
timeline["event_time"] = pd.to_datetime(timeline["event_time"], utc=True)
timeline["recorded_time"] = pd.to_datetime(timeline["recorded_time"], utc=True)
timeline["recording_delay_seconds"] = (
    timeline["recorded_time"] - timeline["event_time"]
).dt.total_seconds()

fig, ax = plt.subplots(figsize=(11, 4), facecolor="#1a1a2e")
ax.set_facecolor("#1a1a2e")
colors = {"FACT": "#1d4ed8", "DECISION": "#b91c1c", "EVIDENCE": "#b45309", "COMMUNICATION": "#15803d"}
for row_number, row in timeline.iterrows():
    ax.scatter(row["event_time"], row_number, color=colors[row["kind"]], s=90)
    ax.hlines(row_number, row["event_time"], row["recorded_time"], color="#e2e8f0", linewidth=2)
ax.set_yticks(range(len(timeline)), timeline["event"])
ax.tick_params(colors="#ffffff")
ax.set_title("Exercise timeline: event time to recorded time", color="#ffffff")
ax.grid(axis="x", alpha=0.2, color="#ffffff")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()
print(f"Maximum authored recording delay: {timeline['recording_delay_seconds'].max():.0f} seconds.")
print("STATIC NOTE: these timestamps are exercise data, not an observed Riverside incident.")

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Paste request bodies and manuscript text into the incident channel | Broadens disclosure and breaks handling controls |
| Right | Store restricted evidence in the approved system and link stable references | Responders investigate without redistributing content |
| Wrong | Rewrite the timeline when a fact changes | Destroys the decision history |
| Right | Append a correction and supersede the earlier statement | Reviewers reconstruct what the team knew at each decision |

**Quick Health Check**

- Each item has a source, collector, method/version, time range, integrity reference, storage location, access list, and retention owner.
- Event time and observation time are distinct.
- Facts, hypotheses, and decisions have different labels.
- Raw customer content, tokens, credentials, and personal data stay out of broad channels.
- Legal hold and evidence-export decisions are routed to authorized owners.

## 3 · Classify Severity from Plausible Impact

Severity is not a label for how alarming the first screenshot looks. It controls command, urgency, review, and communication. Use the highest plausible impact while material uncertainty remains, then downgrade with evidence.

```mermaid
flowchart TD
    A["Known symptom"] --> B{"Confirmed restricted disclosure or active credential abuse?"}
    B -->|Yes| C["SEV-0 posture"]
    B -->|No| D{"False allow, broad outage, unsafe output, deletion failure?"}
    D -->|Yes or materially unknown| E["SEV-1 posture"]
    D -->|No| F{"Bounded degradation, quality, cost, or write failure?"}
    F -->|Yes| G["SEV-2 posture"]
    F -->|No, internal only| H["SEV-3 posture"]
    C --> I["Record evidence needed to downgrade"]
    E --> I
    G --> I
    H --> I
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style I fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** The negative isolation test proves one false allow for restricted EU content. Production blast radius is unknown. Do you start at `SEV-3` because it was a test, `SEV-2` because one request is bounded, or the canonical `SEV-1` posture because a material false allow is proven and scope is uncertain?

In [ ]:
# -- Resolve severity and practice an evidence-based downgrade --------------
severity_rank = {"SEV-0": 0, "SEV-1": 1, "SEV-2": 2, "SEV-3": 3}
canonical_severity = identity_incident["initial_severity"]
print(f"Prediction resolved: start at {canonical_severity}; the false allow is proven and blast radius is unknown.")

# CHANGE THIS only after evidence proves the false allow was isolated and no production path shared the defect.
proposed_severity = "SEV-1"
downgrade_evidence_complete = False
is_unsafe_downgrade = (
    severity_rank[proposed_severity] > severity_rank[canonical_severity]
    and not downgrade_evidence_complete
)
print(f"{'FAIL' if is_unsafe_downgrade else 'PASS'}: severity change has the required evidence.")
print("Severity can fall only when retained evidence narrows plausible impact.")

**Common Pitfalls and Quick Health Check**

| Wrong | Right |
|---|---|
| Severity follows the detector name or affected request count | Severity follows known and plausible confidentiality, integrity, availability, residency, and side-effect impact |
| Downgrade after containment because impact stopped growing | Name the audit query or negative test required to narrow historical scope |

Verify that the severity owner is named, known and plausible impact are separate, unknown blast radius pushes severity upward, downgrade evidence is explicit, and response targets or notification duties are not invented by the notebook.

## 4 · Communicate Safely Under Uncertainty

A useful update is not a forensic dump. It states what is known, unknown, affected, contained, and next. Restricted evidence stays in the approved system.

```mermaid
flowchart LR
    A["Facts and unknowns"] --> B["Draft bounded update"]
    B --> C{"Raw content, identifiers, speculation, or promise?"}
    C -->|Yes| D["Redact or remove"]
    C -->|No| E["Incident command review"]
    D --> B
    E --> F{"Security, privacy, legal, or contract review required?"}
    F -->|Yes| G["Authorized owner review"]
    F -->|No| H["Approve audience and channel"]
    G --> H
    H --> I["Send and retain version"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style I fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** Which sentence is safe for the first customer update?

1. “A stale group cache caused a breach; service will be restored in 30 minutes.”
2. “At 07:25 UTC, a negative access test identified an authorization failure affecting the EU manuscript route. We disabled that route and are establishing scope. The next update is due at 08:00 UTC.”
3. “User API-USER-CONTRACTOR-044 retrieved DOC-MANUSCRIPT-ARIA-037; see the attached trace.”

### Honest organizational and legal boundary

You can establish technical facts, draft a scoped update, and identify systems and data classes. You cannot unilaterally determine breach status, notification law, privilege, regulator wording, contractual remedies, cyber-insurance notice, or customer compensation. Route those decisions to authorized legal, privacy, security, communications, business, and customer owners. A missing owner is an operational readiness gap, not permission to improvise.

In [ ]:
# -- Lint a redacted customer update and practice the structure --------------
safe_update = (
    "At 07:25 UTC, a negative access test identified an authorization failure "
    "affecting the EU manuscript route. We disabled that route and are establishing "
    "scope. The next update is due at 08:00 UTC."
)
unsafe_patterns = {
    "raw user ID": r"API-USER-[A-Z0-9-]+",
    "raw document ID": r"DOC-[A-Z0-9-]+",
    "unsupported root cause": r"root cause|caused by|breach",
    "recovery promise": r"restored in|resolved by|guarantee",
    "credential material": r"bearer\s+|api[_ -]?key|access[_ -]?token",
}
findings = [label for label, pattern in unsafe_patterns.items() if re.search(pattern, safe_update, re.IGNORECASE)]
required_phrases = ["At 07:25 UTC", "disabled that route", "establishing scope", "next update"]
complete = all(phrase.lower() in safe_update.lower() for phrase in required_phrases)
print(f"{'PASS' if not findings and complete else 'FAIL'}: option 2 is bounded and redacted.")

# CHANGE THIS: keep the structure and replace the exercise wording without adding raw IDs.
exercise_update = (
    "At 10:10 UTC, a sandbox regression query returned no current-policy result. "
    "Production impact is not established. We stopped index promotion and pinned the "
    "last-known-good index. The next update is due at 10:45 UTC."
)
exercise_findings = [label for label, pattern in unsafe_patterns.items() if re.search(pattern, exercise_update, re.IGNORECASE)]
exercise_complete = all(phrase in exercise_update.lower() for phrase in ["utc", "not established", "stopped", "next update"])
print(f"{'PASS' if not exercise_findings and exercise_complete else 'FAIL'}: exercise communication structure.")
print("A lexical check supports review; it never replaces human approval or context review.")

**Common Pitfalls and Quick Health Check**

| Wrong | Right |
|---|---|
| Promise a recovery time to sound decisive | State the next update time you control |
| Name a root cause after one correlated change | Label it a hypothesis until a discriminating check supports it |
| Say “no customer impact” before scope review | Say “no additional impact found in the bounded review” |
| Share raw IDs and traces broadly | Link restricted evidence references in the approved system |

Before sending, verify known, unknown, affected, contained, next update, audience, channel, redaction, and approvals. A regular expression catches obvious tokens; a human owner still reviews meaning and obligations.

## 5 · Triage Causally Across Seven Boundaries

The component nearest the symptom is not automatically the cause. A wrong answer can begin with a superseded source, an indexing omission, a missing authorization filter, prompt assembly, model behavior, tool state, or a degraded dependency. Test boundaries in an order that changes the decision.

```mermaid
flowchart TD
    A["Observed failure"] --> B{"Authority decision correct?"}
    B -->|No| C["Policy or identity"]
    B -->|Yes| D{"Source current, unique, authorized, and indexed?"}
    D -->|No| E["Data or retrieval"]
    D -->|Yes| F{"Evidence reached model and answer policy held?"}
    F -->|No| G["Model or orchestration"]
    F -->|Yes| H{"Side effect committed or dependency degraded?"}
    H -->|Committed or ambiguous| I["Tool and workflow state"]
    H -->|Dependency| J["Infrastructure"]
    C --> K["Run discriminating check"]
    E --> K
    G --> K
    I --> K
    J --> K
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style I fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style J fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style K fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Your turn:** The current policy disappears after index promotion. Do not change the model first. Choose the first boundary test that distinguishes source omission, embedding/index omission, filter failure, and answer-policy failure.

In [ ]:
# -- Build the domain matrix and choose a discriminating check ---------------
triage_matrix = pd.DataFrame([
    {
        "exercise_id": scenario["exercise_id"],
        "domain": scenario["domain"],
        "first_question": scenario["causal_questions"][0],
    }
    for scenario in scenarios
]).sort_values("domain")
display(triage_matrix)

# CHANGE THIS: select one named check from the allowed set.
selected_check = "compare_source_index_filter_prompt_ids"
allowed_checks = {
    "compare_source_index_filter_prompt_ids": "Localizes the first boundary where the current policy disappears.",
    "rerun_candidate_query_only": "Reproduces the symptom but does not localize it.",
    "change_model_temperature": "Changes generation noise before proving evidence reached the model.",
}
discriminating = selected_check == "compare_source_index_filter_prompt_ids"
print(f"{'PASS' if discriminating else 'FAIL'}: {allowed_checks[selected_check]}")
print("The first missing stable ID identifies the boundary to repair.")

**Common Pitfalls and Quick Health Check**

| Wrong | Right |
|---|---|
| Change three components and see whether the symptom disappears | Change one bounded variable after one discriminating check |
| Call correlation causation | Record at least one competing hypothesis and a falsifier |
| Retry an ambiguous tool write | Query committed state first, then retry, compensate, or escalate |
| Treat missing telemetry as health | Lower confidence and preserve uncertainty |

A healthy triage record has competing hypotheses for material incidents, one cheap safe check per hypothesis, explicit results, and a supported cause that explains the observed mechanism without claiming more scope than the evidence.

## 6 · Remediate Narrowly and Run Regression Gates

A fix that removes the observed symptom is only a candidate remediation. Recovery evidence must prove the affected behavior, forbidden behavior, adjacent workflows, detection path, and rollback or compensation path.

```mermaid
flowchart LR
    A["Supported cause"] --> B["Smallest remediation"]
    B --> C["Positive behavior gate"]
    C --> D["Negative authorization or policy gate"]
    D --> E["Adjacent workflow gate"]
    E --> F["Telemetry and redaction gate"]
    F --> G["Rollback or compensation drill"]
    G --> H{"All mandatory gates pass?"}
    H -->|No| I["Keep containment"]
    H -->|Yes| J["Request re-enablement review"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style I fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style J fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Your turn:** Add the missing tool-incident gate. The package proves one transition commits and read-only assistance works. It does not prove that the human approved the exact arguments that execute.

In [ ]:
# -- Check gate depth and close the approval-binding gap ---------------------
gate_report = pd.DataFrame([
    {
        "exercise_id": scenario["exercise_id"],
        "domain": scenario["domain"],
        "gate_count": len(scenario["regression_gates"]),
    }
    for scenario in scenarios
])
display(gate_report)
all_have_five = bool((gate_report["gate_count"] >= 5).all())
print(f"{'PASS' if all_have_five else 'FAIL'}: every scenario defines at least five regression gates.")

# CHANGE THIS: write the missing exact-state approval gate.
missing_gate = "Human approval remains bound to exact arguments and state version."
approval_terms = ["approval", "exact arguments", "state version"]
gate_complete = all(term in missing_gate.lower() for term in approval_terms)
print(f"{'PASS' if gate_complete else 'FAIL'}: approval-binding regression gate.")
print("A retry with changed arguments requires new approval, even during an incident.")

**Common Pitfalls and Quick Health Check**

| Wrong | Right |
|---|---|
| Test only the original request | Add forbidden and neighboring slices |
| Call deployment rollback “recovery” | Reconcile or compensate committed external actions separately |
| Pass a gate on a different release, index, or policy | Bind evidence to the exact candidate state |
| Disable alerts to quiet the incident | Prove detection and redaction still work |

Before requesting recovery, verify exact versions, test populations, pass criteria, evidence references, owners, results, containment readiness, and an approved observation window.

## 7 · Re-enable by Evidence and Authority

Passing tests answers “does this candidate satisfy these checks?” It does not answer “who accepts the residual risk for this customer scope?” Re-enablement needs both evidence and authority.

```mermaid
flowchart TD
    A["Regression package"] --> B{"Mandatory gates pass?"}
    B -->|No| C["Hold containment"]
    B -->|Yes| D{"Residual risks owned with expiry?"}
    D -->|No| C
    D -->|Yes| E{"Incident, service, control, and customer approvals?"}
    E -->|No| C
    E -->|Yes| F["Enable bounded cohort"]
    F --> G["Observe named signals"]
    G --> H{"Stop condition fires?"}
    H -->|Yes| C
    H -->|No| I["Next exposure decision"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style I fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** All retrieval regression gates pass, but the editorial workflow owner has not reviewed restored policy behavior. Should the incident commander re-enable because the technical evidence is green?

In [ ]:
# -- Decide whether re-enablement is allowed --------------------------------
def reenable_decision(gates_pass, residual_risks_owned, approvals, required_roles):
    missing_approvals = sorted(set(required_roles) - set(approvals))
    if not gates_pass:
        return "HOLD: mandatory regression gate failed"
    if not residual_risks_owned:
        return "HOLD: residual risk lacks owner or expiry"
    if missing_approvals:
        return f"HOLD: missing approvals from {', '.join(missing_approvals)}"
    return "APPROVE BOUNDED RE-ENABLEMENT"

required_roles = scenario_by_id["IRX-RIV-007"]["reenablement_roles"]
recorded_approvals = [
    "incident commander or delegated retrieval incident owner",
    "retrieval owner",
]
decision = reenable_decision(True, True, recorded_approvals, required_roles)
print(decision)
print("Prediction resolved: technical green is insufficient without the editorial workflow owner.")

**Common Pitfalls and Quick Health Check**

| Wrong | Right |
|---|---|
| Restore full traffic after one smoke test | Start with a bounded cohort and observation window |
| Treat no new alerts as proof of safety | Verify traffic, telemetry, quality, policy, and negative slices |
| Let the implementer accept their own residual risk | Route acceptance to the owner with authority |
| Leave a temporary bypass with no expiry | Name owner, expiry, removal gate, and automatic escalation |

A valid decision records exact candidate state, exposure scope, every mandatory gate, residual risks, approvers, observation signals, stop conditions, next decision time, and rollback or containment owner. Missing evidence or authority means hold.

## 8 · Postmortem: Turn Causes into Verified Change

A blameless review asks which system conditions and decision contexts made the failure possible, hard to detect, or hard to contain. It distinguishes trigger, direct cause, contributing conditions, detection gaps, and lucky limits on impact.

```mermaid
flowchart LR
    A["Verified timeline and impact"] --> B["Trigger"]
    B --> C["Direct cause"]
    C --> D["Contributing conditions"]
    D --> E["Detection and response gaps"]
    E --> F["Owned corrective actions"]
    F --> G{"Completion evidence exists?"}
    G -->|No| H["Action remains open"]
    G -->|Yes| I["Run recurrence test or drill"]
    I --> J{"Control holds?"}
    J -->|No| F
    J -->|Yes| K["Close with residual risk recorded"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style I fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style J fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style K fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Your turn:** Which action is verifiable?

- “Remind engineers to check stale groups.”
- “Add a disabled-user negative test with stale nested-group membership; block release on any false allow; identity owner supplies the retained test report by the due date.”

In [ ]:
# -- Check whether a corrective action can be closed ------------------------
corrective_actions = [
    {
        "action": "Remind engineers to check stale groups",
        "owner": None,
        "due": None,
        "completion_evidence": None,
        "recurrence_check": None,
    },
    {
        "action": "Add disabled-user stale-group negative test and block release on false allow",
        "owner": "identity owner",
        "due": "2026-08-15",
        "completion_evidence": "retained isolation test report",
        "recurrence_check": "release gate returns zero false allows",
    },
]
required_action_fields = ["owner", "due", "completion_evidence", "recurrence_check"]
for action in corrective_actions:
    verifiable = all(action[field] for field in required_action_fields)
    print(f"{'PASS' if verifiable else 'FAIL'}: {action['action']}")
print("The second action can be tested; the reminder cannot prove a control changed.")

**Common Pitfalls and Quick Health Check**

| Wrong | Right |
|---|---|
| Stop at “human error” | Ask which control, interface, review, ownership, or signal allowed one mistake to become impact |
| Force one root cause | Record interacting causes when evidence supports them |
| Close when the document is approved | Close when actions have evidence and recurrence checks hold |
| Ignore what was lucky | Convert uncontrolled limits on impact into explicit controls |

Verify bounded impact; trigger, direct cause, contributing conditions, and detection gaps; blameless language; action-to-gap mapping; owners; due dates; completion evidence; temporary-control expiry; authorized legal/organizational follow-up; and a recurrence drill.

## 9 · Practice, Coverage, and Handoff

The incident is not operationally complete until the receiving team can use the alert, execute containment, find approved evidence, draft a safe update, interpret the gates, and route approval without private FDE context.

```mermaid
flowchart LR
    A["Seeded incident"] --> B["Operator drill"]
    B --> C{"Containment and evidence correct?"}
    C -->|No| D["Repair runbook or training"]
    C -->|Yes| E{"Communication and gates correct?"}
    E -->|No| D
    E -->|Yes| F{"Authority routed correctly?"}
    F -->|No| D
    F -->|Yes| G["Record drill evidence"]
    G --> H["Handoff to customer operations"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

### Final exercise set

1. **Policy:** distinguish lifecycle filtering from ranking and answer-policy failures.
2. **Data:** preserve parser and chunk lineage before removing duplicates.
3. **Retrieval:** localize the first boundary where a current authorized record disappears.
4. **Model:** prove abstention when authorized evidence is absent.
5. **Tool:** reconcile an ambiguous commit before retry or compensation.
6. **Identity:** contain a false allow and prove revocation freshness with negative slices.
7. **Infrastructure:** preserve residency while validating primary-region recovery and approved degraded mode.

In [ ]:
# ── Final Static-Contract Health Check ────────────────────────────────────────
final_checks = {
    "seven domains": {scenario["domain"] for scenario in scenarios} == required_domains,
    "six canonical extensions": sum(scenario["source_kind"] == "canonical_extension" for scenario in scenarios) == 6,
    "one exercise-only derivation": sum(scenario["source_kind"] == "exercise_only_derived" for scenario in scenarios) == 1,
    "evidence fields": all(scenario["evidence_to_preserve"] for scenario in scenarios),
    "causal questions": all(len(scenario["causal_questions"]) >= 3 for scenario in scenarios),
    "regression gates": all(len(scenario["regression_gates"]) >= 5 for scenario in scenarios),
    "reenablement authority": all(scenario["reenablement_roles"] for scenario in scenarios),
}
for label, passed in final_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {label}")

if all(final_checks.values()):
    print("Static fixture contract is complete for later drills.")
else:
    print("Contract gap remains; do not claim exercise completion.")
print("No local result can substitute for customer, security, legal, or production approval.")

## 10 · Completed Roadmap, Coverage Ledger, and Honest Close

```mermaid
flowchart LR
    A["Unsafe path detected"] --> B["Contain first"]
    B --> C["Preserve evidence"]
    C --> D["Classify plausible impact"]
    D --> E["Communicate known and unknown"]
    E --> F["Test causal boundaries"]
    F --> G["Remediate and regress"]
    G --> H["Authorize re-enablement"]
    H --> I["Verify corrective action"]
    style A fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style I fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

### Completed roadmap

- The first action now stops the narrow unsafe path instead of debugging through ongoing exposure.
- Evidence records preserve references, hashes, custody, access, and UTC timeline semantics without copying restricted content into broad channels.
- Severity starts from plausible impact and uncertainty; downgrade requires new evidence rather than optimism.
- Customer updates separate facts, hypotheses, unknowns, containment, safe alternatives, and the next evidence-backed update time.
- Causal triage tests policy, data, retrieval, model, tool, identity, and infrastructure boundaries before changing the most visible component.
- Recovery gates cover positive, negative, adjacent, telemetry, and rollback or committed-action paths.
- Re-enablement remains a new exposure decision requiring scoped evidence, residual-risk acceptance, observation, and named authority.
- Corrective actions close only with an owner, due date, retained completion evidence, and a recurrence check.

### Reflection bridges across the failure chain

| What the step fixed | Residual failure that forced the next step |
|---|---|
| Containment stopped new exposure | Investigation could still leak or destroy evidence |
| Evidence preservation protected provenance | The team could still understate uncertain impact |
| Severity routed response | Updates could still mix facts, hypotheses, identifiers, and promises |
| Safe communication bounded external claims | A plausible component could still be blamed without a discriminating test |
| Causal triage localized the boundary | A symptom patch could still leave forbidden and adjacent paths untested |
| Regression gates tested the candidate fix | Passing tests still did not grant authority to re-enable |
| Re-enablement bound scope and approvers | Temporary controls and detection gaps could still decay without verified follow-through |

### Coverage ledger

| Tier | Coverage | Boundary |
|---|---|---|
| Built and executed against synthetic fixtures | Scenario provenance, containment selection, timeline construction, severity routing, communication linting, causal matrix, gate depth, re-enablement checks, corrective-action completeness | The verified run was cleared and is not retained incident or production evidence |
| Explained and illustrated | Evidence custody, customer communication authority, multi-owner re-enablement, blameless causal review, operator handoff | Requires organizational process and retained drills |
| Named with external validation required | Paging, legal/privacy notification, privilege, regulator/customer wording, live evidence export, production containment, vendor escalation, recovery timing | Requires authorized people, systems, and jurisdiction-specific review |

If a technique named in this notebook is absent from this ledger, treat that as a coverage defect.

### Key takeaways

1. Contain the unsafe path before diagnosing it.
2. Preserve evidence by reference and custody; do not spread restricted content to make investigation convenient.
3. Severity follows plausible impact and uncertainty, not the apparent size of the first sample.
4. Facts, hypotheses, unknowns, and promises belong in different fields.
5. Test the earliest causal boundary that distinguishes competing explanations.
6. Deployment rollback stops exposure; reconciliation determines committed state; correction or compensation requires the matching business authority.
7. Passing tests are evidence for re-enablement, not authority to re-enable.
8. A postmortem action is complete only when the changed control survives a recurrence check.

> **Forward:** carry the alert, runbook, evidence, approval, drill, and corrective-action contracts into `08-handoff-and-customer-success`, where customer operations must perform them without hidden FDE context.